In [1]:
import os
import re
import time
import requests 
import numpy as np
import pandas as pd
import urllib.request
from bs4 import BeautifulSoup

In [2]:
def get_soup(url):
    '''
    Create a soup object of a web url

    Args: 
        url - String (ex. https://basketball-reference.com/)

    Returns:
        soup - BeautifulSoup object
    '''
    try:
        with urllib.request.urlopen(url) as response:
            html = response.read()
    except urllib.error.URLError as e:
        print(f"Error fetching URL: {e.reason}")

    ### create the soup object
    soup = BeautifulSoup(html, 'html.parser')

    return soup

https://www.basketball-reference.com/

Teams:

Eastern Conference:
- ATL — Atlanta Hawks
- BOS — Boston Celtics
- BRK — Brooklyn Nets
- CHI — Chicago Bulls
- CHO — Charlotte Hornets
- CLE — Cleveland Cavaliers
- DET — Detroit Pistons
- IND — Indiana Pacers
- MIA — Miami Heat
- MIL — Milwaukee Bucks
- NYK — New York Knicks
- ORL — Orlando Magic
- PHI — Philadelphia 76ers
- TOR — Toronto Raptors
- WAS — Washington Wizards

Western Conference:
- DAL — Dallas Mavericks
- DEN — Denver Nuggets
- GSW — Golden State Warriors
- HOU — Houston Rockets
- LAC — Los Angeles Clippers
- LAL — Los Angeles Lakers
- MEM — Memphis Grizzlies
- MIN — Minnesota Timberwolves
- NOP — New Orleans Pelicans
- OKC — Oklahoma City Thunder
- PHO — Phoenix Suns
- POR — Portland Trail Blazers
- SAC — Sacramento Kings
- SAS — San Antonio Spurs
- UTA — Utah Jazz


Players:
- Basketball-Reference encodes player names as: /(last initial)/(first 5 letters of last name)(first 2 letters of first name)01.html
  - i.e. `j/jamesle01.html`
  - Names with the same last initial and first name will increment that last digits of the encoded name as they entered into the BR database.

### Player ID Scraping
- Create a function which pulls the player ID Basketball-Reference associates to each player from their team data.

In [3]:
def get_player_ids(team, year):
    url = f'https://www.basketball-reference.com/teams/{team}/{year}.html'

    soup = get_soup(url)

    ids = []
    players = {}

    rows = soup.find("tbody").find_all("tr")

    for row in rows:
        player_cell = row.find("td", {"data-stat": "player"})
        
        if player_cell:
            link = player_cell.find("a")
            
            if link:
                name = link.text
                href = link.get("href")
                split = href.split('/')[-1]
                if '.html' in split:
                    split = split[0:split.find('.')]
                    players[name] = split
                else:
                    players[name] = href
                    print('Check Player List')

    if not players:
        print('No Team Found')
    else:
        # print(ids)
        return players
    
kings = get_player_ids('SAC', 2026)
kings

{'DeMar DeRozan': 'derozde01',
 'Nique Clifford': 'cliffni01',
 'Maxime Raynaud': 'raynama01',
 'Precious Achiuwa': 'achiupr01',
 'Russell Westbrook': 'westbru01',
 'Malik Monk': 'monkma01',
 'Dylan Cardwell': 'cardwdy01',
 'Drew Eubanks': 'eubandr01',
 'Zach LaVine': 'lavinza01',
 'Devin Carter': 'cartede02',
 'Daeqwon Plowden': 'plowdda01',
 'Doug McDermott': 'mcderdo01',
 'Keegan Murray': 'murrake02',
 'Killian Hayes': 'hayeski01',
 'Domantas Sabonis': 'sabondo01',
 'Patrick Baldwin Jr.': 'baldwpa01',
 'Isaiah Stevens': 'steveis01',
 "De'Andre Hunter": 'huntede01'}

In [11]:
def get_player_data(username):
    url = f"https://www.basketball-reference.com/players/{username[0]}/{username}.html"
    soup = get_soup(url)
    tbodies = soup.find_all("tbody")

    tables = {} 

    for i, tbody in enumerate(tbodies):
        
        rows_data = []
        table_name = None
        
        for row in tbody.find_all("tr"):
            row_id = row.get("id")
            
            if row_id and table_name is None:
                table_name = row_id.split(".")[0]
            
            cells = row.find_all(["th", "td"])
            
            row_dict = {}
            for j, cell in enumerate(cells):
                col_name = cell.get("data-stat")
                text = cell.get_text(strip=True)
                
                if col_name is None:
                    col_name = f"col_{j}"
                
                row_dict[col_name] = text
            
            if row_dict:
                rows_data.append(row_dict)

        df = pd.DataFrame(rows_data)

        df['player'] = username
        
        if table_name is None:
            table_name = f"table_{i}"

            if table_name == 'table_0' and df.shape[0] == 5:
                table_name = 'past_5_games'
            elif table_name == 'table_3' and df.shape[0] == 7:
                table_name = 'stathead_insight'
        
        tables[table_name] = df

    for name, df in tables.items():
        print(f"{name}: {df.shape}")
    
    return tables

data = get_player_data('westbru01')

past_5_games: (5, 28)
per_game_stats: (20, 32)
per_game_stats_post: (16, 32)
stathead_insight: (7, 4)
totals_stats: (20, 33)
totals_stats_post: (16, 33)
per_minute_stats: (20, 32)
per_minute_stats_post: (16, 32)
advanced: (20, 30)
advanced_post: (16, 30)
shooting: (20, 32)
shooting_post: (16, 32)


In [10]:
import sqlite3

In [ ]:
with sqlite3.connect("example.db") as conn:
    cur = conn.cursor()

    cur.execute(""" 
        CREATE TABLE IF NOT EXISTS westbrook_past (
            id INTEGER PRIMARY KEY, 
            name TEXT, 
            age INTEGER
        )
    """)

In [29]:
s = ""

for col in data['past_5_games']:
    s = s + f"{col} TEXT,\n"

print(s)

date TEXT,
team_name_abbr TEXT,
game_location TEXT,
opp_name_abbr TEXT,
game_result TEXT,
is_starter TEXT,
mp TEXT,
fg TEXT,
fga TEXT,
fg_pct TEXT,
fg3 TEXT,
fg3a TEXT,
fg3_pct TEXT,
ft TEXT,
fta TEXT,
ft_pct TEXT,
orb TEXT,
drb TEXT,
trb TEXT,
ast TEXT,
stl TEXT,
blk TEXT,
tov TEXT,
pf TEXT,
pts TEXT,
game_score TEXT,
plus_minus TEXT,
player TEXT,

